# GrandNexus v7.5.0 — the living bench, lab version (logbook 79-80), replayed on Colab
**0 API call, no key needed.** Facts arriving over sessions, read by GrandNexus's own grammar pathway; a structured memory
(repaired episodic memory + 2025 graph engine or semantic memory) against text contexts, on 40 never-used worlds, criteria
written before measuring. This notebook replays the measurement: expected, five PASS lines and 100% for the structured pipelines.
Run the cells in order; if a cell prints STOP, send the output. If the Colab assistant offers to rewrite a cell, decline.

## 1. Get exactly v7.5.0

In [ ]:
# 1. Get EXACTLY GrandNexus v7.5.0 (never another zip; stop if it is not found)
import os, glob, fnmatch, subprocess, zipfile, shutil, time
VERSION, ZIP_PATTERN = 'v7.5.0', 'GrandNexus_v7.5.0*.zip'
CONTENT, WORK = '/content', '/content/GrandNexus'
MYDRIVE = '/content/drive/MyDrive'

def gn_version(path):
    """(git describe, has the story-aware bench) of a GrandNexus folder, or (None, False)."""
    if not os.path.isdir(os.path.join(path, '.git')):
        return None, False
    d = subprocess.run('git describe --tags', shell=True, cwd=path, capture_output=True, text=True).stdout.strip()
    bench = os.path.join(path, 'tools', 'bench_learning.py')
    ok = all(os.path.exists(os.path.join(path, 'tools', f)) for f in ('colab_save.py', 'living_lab.py', 'living_bench.py')) and \
         os.path.exists(bench) and '--story-mode' in open(bench, encoding='utf-8').read()
    return d, ok

try:
    from google.colab import drive; drive.mount(os.path.join(CONTENT, 'drive'))
except Exception as e:
    print('Drive not mounted:', e)

# the release zip, newest first: My Drive/GrandNexus and My Drive first, then anywhere in My Drive (3 levels deep);
# any other GrandNexus zip (older versions, session saves) is ignored
found = glob.glob(os.path.join(MYDRIVE, 'GrandNexus', ZIP_PATTERN)) + glob.glob(os.path.join(MYDRIVE, ZIP_PATTERN))
if not found:
    for root, dirs, names in os.walk(MYDRIVE):
        if root[len(MYDRIVE):].count(os.sep) >= 3:
            dirs[:] = []
        found += [os.path.join(root, n) for n in names if fnmatch.fnmatch(n, ZIP_PATTERN)]
found.sort(key=os.path.getmtime)
if found:
    source = found[-1]
else:
    print(f'{ZIP_PATTERN} not found in My Drive: upload it now (any other zip is refused).')
    try:
        from google.colab import files
        up = next(iter(files.upload()))
    except Exception as e:
        raise SystemExit(f'STOP: no {ZIP_PATTERN} on the Drive and no upload possible ({e}).')
    if not fnmatch.fnmatch(os.path.basename(up), ZIP_PATTERN):
        raise SystemExit(f'STOP: {up} is not {ZIP_PATTERN}.')
    source = os.path.join(CONTENT, up)
print('release zip:', source)

# an existing /content/GrandNexus of another version is moved aside, never reused, never deleted
d, ok = gn_version(WORK)
if os.path.exists(WORK) and not (d or '').startswith(VERSION):
    aside = WORK + '_aside_' + time.strftime('%Y%m%d_%H%M%S')
    shutil.move(WORK, aside); print(f'existing {WORK} ({d}) moved aside to {aside}')
if not os.path.exists(WORK):
    z = zipfile.ZipFile(source)
    top = {n.split('/')[0] for n in z.namelist() if n.strip('/')}
    z.extractall(CONTENT if top == {'GrandNexus'} else WORK)
    print('extracted')
subprocess.run('git checkout -q main', shell=True, cwd=WORK)
d, ok = gn_version(WORK)
print('git describe:', d, '| v7.5.0 tools present:', ok)
print(subprocess.run('git log --oneline -1 && git branch', shell=True, cwd=WORK, capture_output=True, text=True).stdout)
if not ((d or '').startswith(VERSION) and ok):
    raise SystemExit(f'STOP: {WORK} is {d}, not {VERSION}. Do not run the next cells.')
print(f'OK: {VERSION}')


## 2. Offline tests (0 calls)

In [ ]:
# 3. Offline tests (0 calls)
import subprocess, sys, glob, os
WORK = '/content/GrandNexus'
bad = []
for t in sorted(glob.glob(os.path.join(WORK, 'tests_lab', 'test_*.py'))):
    r = subprocess.run([sys.executable, '-W', 'ignore', t], cwd=WORK, capture_output=True, text=True)
    last = (r.stdout.strip().splitlines() or ['(no output)'])[-1]
    ok = r.returncode == 0 and not last.startswith('FAIL') and 'Traceback' not in r.stderr
    print(f"{'ok ' if ok else 'BAD'} {os.path.basename(t):40} {last[:70]}")
    if not ok:
        bad.append(t)
if bad:
    raise SystemExit(f'STOP: {len(bad)} offline test(s) failed; send this output.')


## 3. The pre-registered measurement (0 calls, a few minutes)

In [ ]:
# 4. Living bench, lab version, the pre-registered run of logbook 79-80 (0 API call): must print five PASS lines
import subprocess, sys
WORK = '/content/GrandNexus'
p = subprocess.Popen([sys.executable, '-W', 'ignore', 'tools/living_lab.py', '--seed', '7', '--worlds', '40', '--sessions', '12'],
                     cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if 'Warning' not in line:
        sys.stdout.write(line)
p.wait()


## 4. Save to Drive

In [ ]:
# 5. Save to Drive (the repository's own tool: it zips the repository it belongs to and checks the result is inside)
import subprocess, sys
WORK = '/content/GrandNexus'
r = subprocess.run([sys.executable, 'tools/colab_save.py', '--tag', 'living_lab', '--expect', 'tests_lab/resultats/living/living_lab_*.json'],
                   cwd=WORK, capture_output=True, text=True)
print(r.stdout, r.stderr[-500:])
